# 08 - LoRA Fine-tuning

Implement Low-Rank Adaptation (LoRA) for efficient fine-tuning on local field data.

## Why LoRA?
- Pre-trained model has millions of parameters
- Local field data has only 100-500 samples
- Full fine-tuning would overfit
- LoRA adds small trainable matrices (~1% of parameters)

## Reference
- LoRA paper: https://arxiv.org/abs/2106.09685

In [ ]:
import torch
import torch.nn as nn
import math
from pathlib import Path

RESULTS_DIR = Path('../../results/model_experiments')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. LoRA Layer Implementation

In [ ]:
class LoRALayer(nn.Module):
    """
    Low-Rank Adaptation layer.
    
    Instead of fine-tuning W directly, we add:
        W' = W + BA
    where B (d × r) and A (r × k) are low-rank matrices.
    
    This reduces trainable parameters from d×k to (d+k)×r.
    """
    
    def __init__(
        self,
        original_layer: nn.Linear,
        rank: int = 8,
        alpha: float = 16.0,
        dropout: float = 0.05
    ):
        super().__init__()
        
        self.original_layer = original_layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        in_features = original_layer.in_features
        out_features = original_layer.out_features
        
        # Low-rank matrices
        self.lora_A = nn.Parameter(torch.zeros(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        
        # Initialize A with Kaiming, B with zeros (start at original weights)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        
        self.dropout = nn.Dropout(dropout)
        
        # Freeze original weights
        for param in self.original_layer.parameters():
            param.requires_grad = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward with LoRA adaptation."""
        # Original forward
        original_out = self.original_layer(x)
        
        # LoRA forward: x @ A^T @ B^T
        lora_out = self.dropout(x)
        lora_out = lora_out @ self.lora_A.T
        lora_out = lora_out @ self.lora_B.T
        
        return original_out + self.scaling * lora_out
    
    @property
    def n_trainable_params(self) -> int:
        """Count trainable parameters."""
        return self.lora_A.numel() + self.lora_B.numel()

# Test LoRA layer
original = nn.Linear(256, 256)
lora = LoRALayer(original, rank=8, alpha=16)

original_params = sum(p.numel() for p in original.parameters())
lora_params = lora.n_trainable_params

print(f"Original layer parameters: {original_params:,}")
print(f"LoRA trainable parameters: {lora_params:,}")
print(f"Parameter reduction: {100*(1 - lora_params/original_params):.1f}%")

## 2. Apply LoRA to Transformer

In [ ]:
def apply_lora_to_model(
    model: nn.Module,
    target_modules: list = ['query', 'value'],
    rank: int = 8,
    alpha: float = 16.0
):
    """
    Apply LoRA to specified modules in a model.
    
    Args:
        model: PyTorch model
        target_modules: Names of modules to adapt
        rank: LoRA rank
        alpha: LoRA scaling factor
    
    Returns:
        Modified model with LoRA layers
    """
    lora_layers = []
    
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            # Check if module name contains any target
            if any(target in name.lower() for target in target_modules):
                # Replace with LoRA layer
                lora_layer = LoRALayer(module, rank=rank, alpha=alpha)
                
                # Navigate to parent and replace
                parent_name = '.'.join(name.split('.')[:-1])
                child_name = name.split('.')[-1]
                
                if parent_name:
                    parent = model
                    for part in parent_name.split('.'):
                        parent = getattr(parent, part)
                    setattr(parent, child_name, lora_layer)
                else:
                    setattr(model, child_name, lora_layer)
                
                lora_layers.append(name)
    
    print(f"Applied LoRA to {len(lora_layers)} layers:")
    for name in lora_layers:
        print(f"  - {name}")
    
    return model

## 3. Training Configuration for Fine-tuning

In [ ]:
class LoRAConfig:
    """Configuration for LoRA fine-tuning."""
    
    # LoRA parameters
    rank: int = 8
    alpha: float = 16.0
    dropout: float = 0.05
    target_modules: list = ['query', 'value']
    
    # Training
    learning_rate: float = 1e-4
    batch_size: int = 32
    epochs: int = 50
    warmup_steps: int = 100
    weight_decay: float = 0.01
    
    # Early stopping
    patience: int = 10
    min_delta: float = 0.001

lora_config = LoRAConfig()

print("LoRA Fine-tuning Configuration:")
print(f"  Rank: {lora_config.rank}")
print(f"  Alpha: {lora_config.alpha}")
print(f"  Scaling: {lora_config.alpha / lora_config.rank}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"  Learning rate: {lora_config.learning_rate}")

## 4. Fine-tuning Loop Template

In [ ]:
def finetune_with_lora(
    model,
    train_loader,
    val_loader,
    config: LoRAConfig,
    device: str = 'cuda'
):
    """
    Fine-tune model with LoRA on local field data.
    
    Only LoRA parameters are updated.
    """
    model = model.to(device)
    
    # Only optimize LoRA parameters
    lora_params = [
        p for n, p in model.named_parameters() 
        if 'lora_' in n and p.requires_grad
    ]
    
    optimizer = torch.optim.AdamW(
        lora_params,
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.epochs
    )
    
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(config.epochs):
        # Training
        model.train()
        train_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            # Forward (adapt inputs to your model)
            # predictions, _ = model(**batch)
            # loss = criterion(predictions, batch['targets'])
            
            # Placeholder
            loss = torch.tensor(0.0, requires_grad=True)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                # predictions, _ = model(**batch)
                # loss = criterion(predictions, batch['targets'])
                loss = torch.tensor(0.0)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss - config.min_delta:
            best_val_loss = val_loss
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), 'best_lora_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= config.patience:
                print(f"Early stopping at epoch {epoch}")
                break
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    
    return history

print("Fine-tuning template ready")

## 5. Save/Load LoRA Weights

In [ ]:
def save_lora_weights(model, path: Path):
    """Save only LoRA weights (not the full model)."""
    lora_state = {}
    for name, param in model.named_parameters():
        if 'lora_' in name:
            lora_state[name] = param.data
    
    torch.save(lora_state, path)
    print(f"Saved LoRA weights to {path}")
    print(f"  Number of tensors: {len(lora_state)}")
    print(f"  Total parameters: {sum(t.numel() for t in lora_state.values()):,}")


def load_lora_weights(model, path: Path):
    """Load LoRA weights into model."""
    lora_state = torch.load(path)
    
    for name, param in model.named_parameters():
        if name in lora_state:
            param.data = lora_state[name]
    
    print(f"Loaded LoRA weights from {path}")

print("LoRA save/load functions ready")

## 6. Summary

LoRA enables efficient fine-tuning:

| Aspect | Full Fine-tuning | LoRA |
|--------|-----------------|------|
| Parameters | 100% | ~1% |
| Storage | Full model | Adapter only |
| Overfitting risk | High | Low |
| Training time | Long | Short |
| Multi-field support | Separate models | Separate adapters |

In [ ]:
import json

summary = {
    'method': 'LoRA (Low-Rank Adaptation)',
    'reference': 'https://arxiv.org/abs/2106.09685',
    'recommended_config': {
        'rank': 8,
        'alpha': 16,
        'target_modules': ['query', 'value'],
        'learning_rate': 1e-4
    },
    'benefits': [
        '~99% parameter reduction',
        'Prevents overfitting on small datasets',
        'Fast training (~minutes vs hours)',
        'Easy multi-field management (one adapter per field)'
    ]
}

with open(RESULTS_DIR / 'lora_config.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("LoRA configuration saved")